# Lecture 13: Attention

*Notebook for UMich EECS 498/598 (Justin Johnson), Lecture 13 "Attention" (Fall 2020 slides).*

**Attention** started as a fix for one problem in RNN translation models. It turned out to be so useful that it replaced RNNs almost everywhere: it's the core of the **Transformer**, the architecture behind BERT, GPT, and Vision Transformers.

**What you'll learn**
1. The **bottleneck** problem in sequence-to-sequence RNNs.
2. **Attention** for translation and for image captioning, computed by hand.
3. The general **attention layer**: queries, keys, values, and why we scale by $\sqrt{D}$.
4. **Self-attention**: permutation equivariance, positional encodings, masking, multiple heads.
5. **Train** a tiny attention model and watch what it attends to.
6. The **Transformer** block, and how big Transformers have become.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(scores, axis=-1):
    shifted = scores - scores.max(axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=axis, keepdims=True)

## 1. Recap: sequence-to-sequence with RNNs

**Translation:** input "we are eating bread" → output "estamos comiendo pan".
* An **encoder** RNN reads the input: $h_t = f_W(x_t, h_{t-1})$.
* From its **final** hidden state we get the decoder's initial state $s_0$ and a **context vector** $c$ (often $c = h_T$).
* A **decoder** RNN writes the output one word at a time: $s_t = g_U(y_{t-1}, s_{t-1}, c)$, starting from [START] and ending at [STOP].

**Problem:** the whole input sentence is squeezed through **one fixed-size vector** $c$. That's fine for 4 words. **What if T = 1000?**

**Idea:** use a **new context vector at each step** of the decoder, one that looks at whichever input words are relevant right now.

## 2. Seq2seq with attention (Bahdanau et al., 2015)

At decoder step $t$:
1. Compute an **alignment score** between the previous decoder state and **every** encoder hidden state: $e_{t,i} = f_{att}(s_{t-1}, h_i)$. (The paper uses a small MLP for $f_{att}$; here we'll use a **dot product**, which is what later models use.)
2. **Softmax** the scores to get **attention weights**: $0 < a_{t,i} < 1$ and $\sum_i a_{t,i} = 1$.
3. The context vector is a **weighted average** of the hidden states: $c_t = \sum_i a_{t,i}\,h_i$.
4. Use it in the decoder: $s_t = g_U(y_{t-1}, s_{t-1}, c_t)$.

Everything is differentiable, so we **don't supervise the attention weights**: the network learns where to look by backprop.

### By hand
Pretend the encoder produced these hidden states, with dimensions that roughly mean "pronoun", "verb to be", "food action", "food object":

In [ ]:
words = ["we", "are", "eating", "bread"]
h = np.array([[2.0, 0.5, 0.0, 0.0],      # we
              [0.5, 2.0, 0.0, 0.0],      # are
              [0.0, 0.3, 2.0, 0.5],      # eating
              [0.0, 0.0, 0.5, 2.0]])     # bread

def attend(decoder_state, encoder_states):
    scores = encoder_states @ decoder_state        # alignment scores e_i (dot products)
    weights = softmax(scores)                      # attention weights a_i
    context = weights @ encoder_states             # c = sum_i a_i h_i
    return scores, weights, context

s0 = np.array([1.0, 1.0, 0.0, 0.0])               # decoder state while producing "estamos" (= "we are")
scores, weights, context = attend(s0, h)
print("alignment scores: ", {w: float(x) for w, x in zip(words, scores.round(2))})
print("attention weights:", {w: float(x) for w, x in zip(words, weights.round(2))})
print("context vector:   ", context.round(2))

"Estamos" means "we are", so most of the attention goes to those two words. For the next word, "comiendo" ("eating"), the decoder state is different, so the attention moves:

In [ ]:
s1 = np.array([0.0, 0.0, 1.5, 0.3])               # decoder state while producing "comiendo"
_, weights, _ = attend(s1, h)
print("attention weights:", {w: float(x) for w, x in zip(words, weights.round(2))})

**A different context vector at each step:** the input isn't bottlenecked through one vector, and each step "looks at" different parts of the input. Plotting $a_{t,i}$ for real translations shows **diagonal** patterns when the words correspond in order, and off-diagonal ones where the languages order words differently ("European Economic Area" → "zone économique européenne").

**An important observation:** the decoder never uses the fact that the $h_i$ form an **ordered** sequence; it treats them as an unordered **set**. So we can use the same idea on **any** set of vectors, for example a grid of image features.

## 3. Image captioning with attention ("Show, Attend, and Tell", Xu et al., 2015)

Use a CNN to compute a **grid of features** $h_{i,j}$ for the image. At each decoder step, compute an attention weight for **every grid position**, and take the weighted average as the context vector. Each word looks at a different part of the image: "cat" looks at the cat, "outside" at the background.

(This is a bit like human vision: only the tiny **fovea** of the retina sees sharply, so our eyes constantly jump around ("**saccades**") to look at different parts of a scene.)

In [ ]:
rng = np.random.default_rng(0)
grid = rng.normal(0, 0.3, size=(3, 3, 4))           # a 3x3 grid of 4-dim CNN features
grid[0, 1] += [3, 0, 0, 0]                           # "cat-like" features at the top middle
grid[2, :] += [0, 0, 3, 0]                           # "grass-like" features along the bottom row

for word, state in [("cat", np.array([1.0, 0, 0, 0])), ("grass", np.array([0, 0, 1.0, 0]))]:
    scores = grid.reshape(9, 4) @ state              # one score per grid position
    weights = softmax(scores).reshape(3, 3)
    print(f"attention while writing '{word}':\n{weights.round(2)}")

The same pattern ("X, attend, and Y") was applied everywhere: **ask, attend, and answer** (visual question answering), **listen, attend, and spell** (speech recognition), **listen, attend, and walk** (robot navigation), **show, attend, and read** (reading text in images).

## 4. The attention layer

Let's turn this into a general-purpose layer, step by step, exactly as in the lecture.

**Version 1.** Inputs: a **query** vector $q$ (shape $D_Q$), **input vectors** $X$ (shape $N_X\times D_X$), and a similarity function $f_{att}$.
* similarities $e_i = f_{att}(q, X_i)$ → weights $a = \text{softmax}(e)$ → output $y = \sum_i a_iX_i$.

**Change 1: use the dot product for similarity**: $e_i = q\cdot X_i$.

**Change 2: use the *scaled* dot product**: $e_i = q\cdot X_i/\sqrt{D_Q}$.

**Why scale?** If $q$ and $X_i$ have $D$ random entries, their dot product grows like $\sqrt{D}$. Large similarities make the softmax **saturate** (nearly one-hot), and a saturated softmax has **vanishing gradients** (Lecture 10). Let's measure it with $D = 512$:

In [ ]:
D = 512
q = rng.normal(size=D)
X = rng.normal(size=(10, D))

def softmax_gradient_size(e):
    # Size of the softmax's Jacobian: how much the weights can respond to changes in the scores.
    a = softmax(e)
    jacobian = np.diag(a) - np.outer(a, a)
    return np.abs(jacobian).sum()

for name, e in [("unscaled  q.x      ", X @ q), ("scaled    q.x / sqrt(D)", X @ q / np.sqrt(D))]:
    print(f"{name}: scores have std {e.std():5.1f}, largest weight {softmax(e).max():.3f}, "
          f"softmax gradient size {softmax_gradient_size(e):.4f}")

Unscaled, the softmax puts essentially all its weight on one input, and its gradient is almost zero, so learning stalls. Scaling by $\sqrt{D}$ keeps the scores at a reasonable size.

**Change 3: multiple query vectors.** With queries $Q$ (shape $N_Q\times D_Q$), compute everything at once:

$$E = \frac{QX^\top}{\sqrt{D_Q}}\ \ (N_Q\times N_X), \qquad A = \text{softmax}(E, \text{dim}=1), \qquad Y = AX\ \ (N_Q\times D_X)$$

**Change 4: separate keys and values.** The input vectors are used for two different jobs: being **compared** with the query, and being **averaged** into the output. Give each job its own learnable projection:
* **keys** $K = XW_K$ (compared with the queries),
* **values** $V = XW_V$ (averaged into the output).

$$E = \frac{QK^\top}{\sqrt{D_Q}}, \qquad A = \text{softmax}(E, \text{dim}=1), \qquad Y = AV$$

In [ ]:
def attention(Q, K, V):
    # Q: (N_Q, D_Q)  K: (N_X, D_Q)  V: (N_X, D_V)   ->   Y: (N_Q, D_V), A: (N_Q, N_X)
    E = Q @ K.T / np.sqrt(Q.shape[1])        # similarities between every query and every key
    A = softmax(E, axis=1)                   # each query's weights sum to 1
    Y = A @ V                                # each output is a weighted average of the values
    return Y, A

X = rng.normal(size=(3, 6))                  # 3 input vectors, D_X = 6
Q = rng.normal(size=(4, 5))                  # 4 queries, D_Q = 5
W_K, W_V = rng.normal(size=(6, 5)), rng.normal(size=(6, 7))
Y, A = attention(Q, X @ W_K, X @ W_V)
print("output shape:", Y.shape, "  attention weights shape:", A.shape)
print("each row of A sums to 1:", A.sum(axis=1).round(6))

## 5. Self-attention

A **self-attention** layer has **one query per input vector**: the queries also come from the inputs, $Q = XW_Q$. Every input vector looks at every input vector (including itself).

In [ ]:
def self_attention(X, W_Q, W_K, W_V, causal_mask=False):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    E = Q @ K.T / np.sqrt(Q.shape[1])
    if causal_mask:
        N = len(X)
        E = E + np.triu(np.full((N, N), -np.inf), k=1)    # block every "look into the future" entry
    A = softmax(E, axis=1)
    return A @ V, A

D_X, D_Q = 4, 4
W_Q, W_K, W_V = [rng.normal(size=(D_X, D_Q)) for _ in range(3)]
X = rng.normal(size=(5, D_X))                              # a sequence of 5 vectors
Y, A = self_attention(X, W_Q, W_K, W_V)
print("5 vectors in -> 5 vectors out:", Y.shape)

### Self-attention doesn't know about order
**Consider permuting the input vectors.** The queries, keys, and values get permuted the same way, so the similarities, the attention weights, and the outputs are all **the same, just permuted**. Self-attention is **permutation equivariant**: $f(\text{shuffle}(x)) = \text{shuffle}(f(x))$. It works on **sets** of vectors.

In [ ]:
perm = np.array([3, 0, 4, 1, 2])
Y_of_shuffled, _ = self_attention(X[perm], W_Q, W_K, W_V)
print("self_attention(shuffle(X)) == shuffle(self_attention(X)):", np.allclose(Y_of_shuffled, Y[perm]))

That's a problem for language: "dog bites man" and "man bites dog" would look the same. **The fix: add a positional encoding** $E(t)$ to each input, a vector that depends on the position $t$. It can be a learned lookup table, or a fixed function, like the sine waves from the original Transformer paper:

In [ ]:
def positional_encoding(n_positions, dim):
    positions = np.arange(n_positions)[:, None]
    frequencies = 1 / (10000 ** (np.arange(0, dim, 2) / dim))
    encoding = np.zeros((n_positions, dim))
    encoding[:, 0::2] = np.sin(positions * frequencies)
    encoding[:, 1::2] = np.cos(positions * frequencies)
    return encoding

P = positional_encoding(5, D_X)
Y_pos, _ = self_attention(X + P, W_Q, W_K, W_V)
Y_pos_shuffled, _ = self_attention(X[perm] + P, W_Q, W_K, W_V)       # same vectors, different positions
print("with positional encoding, shuffling still just permutes the outputs?", np.allclose(Y_pos_shuffled, Y_pos[perm]))

plt.figure(figsize=(6, 2.5))
plt.imshow(positional_encoding(50, 16).T, aspect="auto", cmap="RdBu")
plt.xlabel("position"); plt.ylabel("dimension"); plt.title("sinusoidal positional encodings")
plt.show()

Now the layer knows where each vector is.

### Masked self-attention
For **language modeling** (predict the next word), a vector must **not look ahead** at words it's supposed to predict. Set the similarities for "future" positions to $-\infty$ before the softmax, so their weights become exactly 0:

In [ ]:
Y_masked, A_masked = self_attention(X, W_Q, W_K, W_V, causal_mask=True)
print("masked attention weights (row = query position):\n", A_masked.round(2))

X_changed = X.copy()
X_changed[4] += 10                                        # change only the LAST input
Y_changed, _ = self_attention(X_changed, W_Q, W_K, W_V, causal_mask=True)
print("outputs 0-3 unchanged when the last input changes:", np.allclose(Y_masked[:4], Y_changed[:4]))

### Multi-head self-attention
Use $H$ independent **attention heads** in parallel, each with its own $W_Q, W_K, W_V$ (and a smaller dimension), then **concatenate** their outputs and mix them with one more matrix $W_O$. Different heads can learn to look for different things (e.g., one tracks the previous word, another the subject of the sentence). Hyperparameters: the query dimension $D_Q$ and the number of heads $H$.

In [ ]:
def multihead_self_attention(X, heads, W_O, causal_mask=False):
    # heads: a list of (W_Q, W_K, W_V), one per head.
    outputs = [self_attention(X, W_Q, W_K, W_V, causal_mask)[0] for W_Q, W_K, W_V in heads]
    return np.concatenate(outputs, axis=1) @ W_O

D, H = 8, 2
heads = [[rng.normal(size=(D, D // H)) for _ in range(3)] for _ in range(H)]   # each head works in D/H dims
W_O = rng.normal(size=(D, D))
X8 = rng.normal(size=(5, D))
print("multi-head output shape:", multihead_self_attention(X8, heads, W_O).shape)

**Self-attention also works on images.** Treat the $H\times W$ grid of CNN features as a set of $HW$ vectors, compute queries, keys, and values with 1×1 convs, let every position attend to every position, and add the result back with a residual connection (e.g., SAGAN, Zhang et al., 2018).

## 6. Let's train an attention model

**Task ("lookup"):** each example is a small dictionary of 6 (**key**, **value**) pairs plus a **query**. The keys are symbols (one of 10), the values are random 4-d vectors. The answer is **the value whose key matches the query**.

**Model:** one attention layer. The query symbol becomes $q = \text{onehot}(\text{query})\,W_Q$; each key becomes $k_i = \text{onehot}(\text{key}_i)\,W_K$; the values are used as-is; the output is $y = \sum_i a_iv_i$. We learn $W_Q$ and $W_K$. If it works, the model has learned **"attend to the matching key"** on its own.

In [ ]:
N_PAIRS, N_SYMBOLS, D_VALUE, D_KEY = 6, 10, 4, 8

def make_lookup_batch(batch_size, rng):
    keys = np.array([rng.choice(N_SYMBOLS, N_PAIRS, replace=False) for _ in range(batch_size)])   # (B, 6)
    values = rng.normal(size=(batch_size, N_PAIRS, D_VALUE))                                      # (B, 6, 4)
    answer_slot = rng.integers(0, N_PAIRS, batch_size)                                            # which pair is asked for
    queries = keys[np.arange(batch_size), answer_slot]
    targets = values[np.arange(batch_size), answer_slot]
    return keys, values, queries, targets, answer_slot

def lookup_forward(W_Q, W_K, keys, values, queries):
    eye = np.eye(N_SYMBOLS)
    q = eye[queries] @ W_Q                                  # (B, 8)
    K = eye[keys] @ W_K                                     # (B, 6, 8)
    E = (K @ q[:, :, None])[:, :, 0] / np.sqrt(D_KEY)       # (B, 6): one score per pair
    A = softmax(E, axis=1)
    y = (A[:, :, None] * values).sum(axis=1)                # (B, 4): weighted average of the values
    return y, A, q, K

The backward pass goes through the weighted average, the softmax, and the dot products. For the softmax, if $d_a$ is the gradient arriving at the weights, the gradient at the scores is $a\odot(d_a - \sum_i a_i\,d_{a,i})$.

In [ ]:
def lookup_loss_and_gradients(W_Q, W_K, keys, values, queries, targets):
    B = len(keys)
    y, A, q, K = lookup_forward(W_Q, W_K, keys, values, queries)
    loss = np.mean(np.sum((y - targets) ** 2, axis=1))
    dy = 2 * (y - targets) / B                                       # (B, 4)
    dA = (values @ dy[:, :, None])[:, :, 0]                          # (B, 6): back through the weighted average
    dE = A * (dA - (A * dA).sum(axis=1, keepdims=True))             # back through the softmax
    dE = dE / np.sqrt(D_KEY)
    dq = (dE[:, :, None] * K).sum(axis=1)                            # back through the dot products
    dK = dE[:, :, None] * q[:, None, :]
    eye = np.eye(N_SYMBOLS)
    dW_Q = eye[queries].T @ dq
    dW_K = eye[keys].reshape(-1, N_SYMBOLS).T @ dK.reshape(-1, D_KEY)
    return loss, dW_Q, dW_K

In [ ]:
# gradient check on one weight
rng = np.random.default_rng(1)
W_Q, W_K = rng.normal(0, 0.5, (N_SYMBOLS, D_KEY)), rng.normal(0, 0.5, (N_SYMBOLS, D_KEY))
batch = make_lookup_batch(16, rng)
_, dW_Q, _ = lookup_loss_and_gradients(W_Q, W_K, *batch[:4])
old = W_Q[2, 3]
W_Q[2, 3] = old + 1e-6; plus = lookup_loss_and_gradients(W_Q, W_K, *batch[:4])[0]
W_Q[2, 3] = old - 1e-6; minus = lookup_loss_and_gradients(W_Q, W_K, *batch[:4])[0]
W_Q[2, 3] = old
print(f"dW_Q[2,3]: backprop {dW_Q[2, 3]:.6f}, numerical {(plus - minus) / 2e-6:.6f}")

Train with Adam (Lecture 4):

In [ ]:
params = {"W_Q": rng.normal(0, 0.5, (N_SYMBOLS, D_KEY)), "W_K": rng.normal(0, 0.5, (N_SYMBOLS, D_KEY))}
m = {k: np.zeros_like(v) for k, v in params.items()}
v_ = {k: np.zeros_like(v) for k, v in params.items()}
losses = []
for t in range(1, 2001):
    keys, values, queries, targets, _ = make_lookup_batch(64, rng)
    loss, dW_Q, dW_K = lookup_loss_and_gradients(params["W_Q"], params["W_K"], keys, values, queries, targets)
    losses.append(loss)
    for name, grad in [("W_Q", dW_Q), ("W_K", dW_K)]:
        m[name] = 0.9 * m[name] + 0.1 * grad
        v_[name] = 0.999 * v_[name] + 0.001 * grad ** 2
        params[name] -= 0.05 * (m[name] / (1 - 0.9 ** t)) / (np.sqrt(v_[name] / (1 - 0.999 ** t)) + 1e-8)
print(f"loss: {np.mean(losses[:20]):.3f} at the start -> {np.mean(losses[-100:]):.1e} at the end")

Test on new dictionaries. As a baseline, compare with a model that can't attend and just **averages all the values** (the "one fixed-size summary" approach):

In [ ]:
keys, values, queries, targets, answer_slot = make_lookup_batch(2000, np.random.default_rng(99))
y, A, _, _ = lookup_forward(params["W_Q"], params["W_K"], keys, values, queries)
average_of_values = values.mean(axis=1)

print(f"attention model: error {np.mean(np.sum((y - targets) ** 2, axis=1)):.1e}, "
      f"attends most to the right pair {np.mean(A.argmax(axis=1) == answer_slot):.1%} of the time")
print(f"average of all values (no attention): error {np.mean(np.sum((average_of_values - targets) ** 2, axis=1)):.4f}")
print("\none example:")
print("  keys:   ", keys[0], "  query:", queries[0])
print("  weights:", A[0].round(3))

**What we see:** the model learned to put almost all of its attention on the pair whose key matches the query, without ever being told which pair was right. (Look at $W_QW_K^\top$: it has become a large diagonal, "symbol $i$ matches symbol $i$".)

In [ ]:
plt.figure(figsize=(3.5, 3))
plt.imshow(params["W_Q"] @ params["W_K"].T, cmap="viridis")
plt.xlabel("key symbol"); plt.ylabel("query symbol"); plt.title("learned W_Q W_K^T")
plt.colorbar(); plt.show()

## 7. Three ways of processing sequences

| | Recurrent neural network | 1-D convolution | Self-attention |
|---|---|---|---|
| Works on | ordered sequences | multidimensional grids | **sets** of vectors |
| Long sequences | (+) after one layer, $h_T$ "sees" the whole sequence | (−) need to stack many layers for outputs to see the whole sequence | (+) after one layer, **every** output sees **all** inputs |
| Parallel? | (−) hidden states must be computed one after another | (+) every output can be computed in parallel | (+) highly parallel: a few matrix multiplies |
| Cost | | | (−) very memory intensive: $N\times N$ attention weights |

(On the slide, self-attention's "good at long sequences" is marked with a (−); it's a typo, it should be (+).)

How fast does that $N\times N$ attention matrix grow?

In [ ]:
for N in [512, 4096, 32768]:
    print(f"sequence length {N:6d}: attention matrix has {N * N:>13,d} entries = {N * N * 4 / 1e9:7.2f} GB per head (32-bit)")

**"Attention is all you need"** (Vaswani et al., 2017): the **Transformer** uses only self-attention to mix information between vectors.

## 8. The Transformer

**A Transformer block:** input a set of vectors $x$, output a set of vectors $y$:
1. **Self-attention** (multi-head), plus a **residual connection**, then **layer normalization**.
2. An **MLP applied to each vector independently**, plus a residual connection, then layer normalization.

**Self-attention is the only interaction between vectors**; layer norm and the MLP work on each vector separately. That makes Transformers highly scalable and parallelizable. **A Transformer is a sequence of these blocks.**

**Layer normalization** (Lecture 7) normalizes each vector by its own mean and standard deviation, then applies a learned scale $\gamma$ and shift $\beta$.

In [ ]:
def layer_norm(H, gamma, beta, eps=1e-5):
    mean = H.mean(axis=1, keepdims=True)                  # one mean per vector
    std = np.sqrt(H.var(axis=1, keepdims=True) + eps)
    return gamma * (H - mean) / std + beta

def transformer_block(X, p):
    attended = multihead_self_attention(X, p["heads"], p["W_O"])
    H = layer_norm(X + attended, p["gamma1"], p["beta1"])           # residual + layer norm
    mlp = np.maximum(0, H @ p["W1"] + p["b1"]) @ p["W2"] + p["b2"]   # the same MLP on each vector
    return layer_norm(H + mlp, p["gamma2"], p["beta2"])              # residual + layer norm

def init_block(D, H, rng):
    s = 1 / np.sqrt(D)
    return {"heads": [[rng.normal(0, s, (D, D // H)) for _ in range(3)] for _ in range(H)],
            "W_O": rng.normal(0, s, (D, D)), "W1": rng.normal(0, s, (D, 4 * D)), "b1": np.zeros(4 * D),
            "W2": rng.normal(0, 1 / np.sqrt(4 * D), (4 * D, D)), "b2": np.zeros(D),
            "gamma1": np.ones(D), "beta1": np.zeros(D), "gamma2": np.ones(D), "beta2": np.zeros(D)}

blocks = [init_block(16, 4, rng) for _ in range(3)]
X16 = rng.normal(size=(7, 16))
out = X16
for block in blocks:
    out = transformer_block(out, block)
print("a 3-block Transformer: 7 vectors of size 16 in ->", out.shape, "out")

out_shuffled = X16[perm.tolist() + [5, 6]]
for block in blocks:
    out_shuffled = transformer_block(out_shuffled, block)
print("still permutation equivariant (no positional encoding):", np.allclose(out_shuffled, out[perm.tolist() + [5, 6]]))

(Modern Transformers move the layer norm **before** the attention and MLP, "pre-norm", which trains more stably. That's covered in CS231n's version of this lecture.)

### Transfer learning: "the ImageNet moment for NLP"
* **Pretraining:** download a lot of text from the internet, and train a giant Transformer as a **language model**.
* **Fine-tuning:** fine-tune it on your own NLP task (BERT, Devlin et al., 2018).

### Scaling up
How many parameters does a Transformer have? Per block: attention has $W_Q, W_K, W_V, W_O$, four $D\times D$ matrices ($4D^2$), and the MLP has $D\times4D$ and $4D\times D$ ($8D^2$). So **each block has about $12D^2$ parameters**, plus the word embeddings (vocabulary size × $D$). Let's check this rule against the lecture's table:

In [ ]:
models = [  # name, layers, width D, vocabulary size, reported parameters
    ("BERT-Base", 12, 768, 30522, "110M"),
    ("BERT-Large", 24, 1024, 30522, "340M"),
    ("GPT-2", 48, 1600, 50257, "1.5B"),
    ("Megatron-LM", 72, 3072, 50257, "8.3B"),
    ("GPT-3", 96, 12288, 50257, "175B"),
]
print(f"{'model':12s} {'12 x layers x D^2 + vocab x D':>30s} {'reported':>10s}")
for name, layers, D, vocab, reported in models:
    estimate = 12 * layers * D ** 2 + vocab * D
    print(f"{name:12s} {estimate / 1e9:27.2f} B {reported:>10s}")

The simple rule is within a few percent every time. Training GPT-3-class models needs thousands of GPUs; Megatron-LM alone was estimated at around $430,000 of cloud compute. The resulting models can **write articles**, **fix grammar from a few examples in the prompt** ("programming by prompt"), and **imitate literary styles**.

(About the slide "Vaswani et al: 12 blocks, $D_Q$ = 512, 6 heads": the table on the next slide lists Transformer-Base with **8** heads, which matches the paper.)

## 9. In PyTorch (optional, run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch
import torch, torch.nn as nn, torch.nn.functional as F

x = torch.randn(1, 10, 64)                                          # (batch, sequence length, D)
attn = nn.MultiheadAttention(embed_dim=64, num_heads=8, batch_first=True)
y, weights = attn(x, x, x)                                          # self-attention: queries = keys = values = x
print("multi-head self-attention:", tuple(y.shape), " weights:", tuple(weights.shape))

causal = F.scaled_dot_product_attention(x, x, x, is_causal=True)    # masked attention in one call
block = nn.TransformerEncoderLayer(d_model=64, nhead=8, batch_first=True)
print("Transformer block output:", tuple(block(x).shape))

## Summary
* **Seq2seq RNNs** squeeze the input through one vector; **attention** builds a new context vector at every decoder step, as a weighted average of all encoder states.
* Attention treats its inputs as a **set**, so it works on image grids too (image captioning).
* **Attention layer:** $E = QK^\top/\sqrt{D_Q}$, $A = \text{softmax}(E)$, $Y = AV$. Scaling keeps the softmax from saturating.
* **Self-attention:** queries, keys, and values all come from the inputs. It's **permutation equivariant**, so add **positional encodings**; **mask** it to stop it from looking ahead; use **multiple heads**.
* **Transformer block:** self-attention + residual + layer norm, then a per-vector MLP + residual + layer norm. About $12D^2$ parameters per block.
* **Pretrain, then fine-tune**, and scale up: BERT, GPT-2, GPT-3.

### Check your understanding
1. What's the bottleneck in a seq2seq RNN, and how does attention remove it?
2. Why divide the similarities by $\sqrt{D_Q}$?
3. What happens to a self-attention layer's outputs if you shuffle its inputs? How do Transformers know the order of words?
4. How does masked self-attention prevent a language model from cheating?
5. Roughly how many parameters does a Transformer with 24 blocks and $D = 1024$ have (ignoring embeddings)?

<details><summary>Answers</summary>

1. The whole input is summarized by one fixed-size vector; attention lets every decoder step look directly at all the encoder states.
2. Dot products of $D$-dimensional vectors grow like $\sqrt{D}$; large scores saturate the softmax and kill its gradients.
3. The outputs are shuffled the same way (permutation equivariance); positional encodings are added to tell the model where each word is.
4. It sets the scores for future positions to $-\infty$, so their attention weights are exactly 0.
5. About $12 \times 24 \times 1024^2 \approx 302$ million (BERT-Large).
</details>

**Next:** Lecture 14, visualizing and understanding CNNs.